# Class 11 — Pipeline benchmark (CSP+LDA vs Riemannian)

Comparing the CSP+LDA decoder against two Riemannian-geometry approaches — MDM (Minimum Distance to Mean) and Tangent Space + LDA — which classify directly on the full channel covariance structure instead of reducing to a handful of spatial components first. Benchmarked across all 9 subjects.

## Setup

In [1]:
import numpy as np
import pandas as pd
import torch
import mne
import matplotlib.pyplot as plt

from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery

from sklearn.pipeline import Pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import cross_val_score, GroupKFold

from mne.decoding import CSP

from pyriemann.estimation import Covariances
from pyriemann.classification import MDM, TSClassifier

mne.set_log_level("WARNING")
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)


/Users/vicky/Documents/motor_imagery_decoding/motor-imagery-decoding/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
paradigm = MotorImagery(n_classes=4, fmin=8, fmax=30, tmin=0.5, tmax=3.5)

csp_lda = Pipeline([
    ("CSP", CSP(n_components=6, reg="ledoit_wolf", log=True, norm_trace=False)),
    ("LDA", LinearDiscriminantAnalysis()),
])

mdm = Pipeline([
    ("Cov", Covariances(estimator="lwf")),
    ("MDM", MDM()),
])

ts_lda = Pipeline([
    ("Cov", Covariances(estimator="lwf")),
    ("TS", TSClassifier()),
])


Choosing from all possible events


## Sanity check on a single subject

In [3]:
dataset = BNCI2014_001()
dataset.subject_list = [1]
X, y, metadata = paradigm.get_data(dataset=dataset, subjects=[1])

groups = metadata["session"]
cv = GroupKFold(n_splits=2)

scores_mdm = cross_val_score(mdm, X, y, cv=cv, groups=groups)
print("MDM:", scores_mdm, scores_mdm.mean())

scores_ts = cross_val_score(ts_lda, X, y, cv=cv, groups=groups)
print("TS+LDA:", scores_ts, scores_ts.mean())


MDM: [0.73263889 0.74305556] 0.7378472222222222
TS+LDA: [0.88194444 0.79166667] 0.8368055555555556


## Full benchmark — 9 subjects x 3 pipelines

In [4]:
results = []
pipelines = {
    "CSP+LDA": csp_lda,
    "MDM": mdm,
    "TS+LDA": ts_lda,
}

for subject in range(1, 10):
    dataset = BNCI2014_001()
    dataset.subject_list = [subject]
    X, y, metadata = paradigm.get_data(dataset=dataset, subjects=[subject])
    groups = metadata["session"]
    cv = GroupKFold(n_splits=2)

    for name, pipe in pipelines.items():
        scores = cross_val_score(pipe, X, y, cv=cv, groups=groups)
        results.append({"subject": subject, "pipeline": name, "mean_acc": scores.mean()})

df_results = pd.DataFrame(results)
print(df_results)


    subject pipeline  mean_acc
0         1  CSP+LDA  0.769097
1         1      MDM  0.737847
2         1   TS+LDA  0.836806
3         2  CSP+LDA  0.531250
4         2      MDM  0.451389
5         2   TS+LDA  0.520833
6         3  CSP+LDA  0.779514
7         3      MDM  0.687500
8         3   TS+LDA  0.842014
9         4  CSP+LDA  0.559028
10        4      MDM  0.510417
11        4   TS+LDA  0.612847
12        5  CSP+LDA  0.395833
13        5      MDM  0.338542
14        5   TS+LDA  0.413194
15        6  CSP+LDA  0.467014
16        6      MDM  0.414931
17        6   TS+LDA  0.529514
18        7  CSP+LDA  0.590278
19        7      MDM  0.482639
20        7   TS+LDA  0.756944
21        8  CSP+LDA  0.725694
22        8      MDM  0.729167
23        8   TS+LDA  0.796875
24        9  CSP+LDA  0.652778
25        9      MDM  0.663194
26        9   TS+LDA  0.777778


## Summary

In [5]:
summary = df_results.groupby("pipeline")["mean_acc"].agg(["mean", "std"])
print(summary)


              mean       std
pipeline                    
CSP+LDA   0.607832  0.134499
MDM       0.557292  0.148919
TS+LDA    0.676312  0.159503


**Result:**

| Pipeline | Mean | Std |
|---|---|---|
| CSP+LDA | 0.608 | 0.134 |
| MDM | 0.557 | 0.149 |
| TS+LDA | **0.676** | 0.160 |

TS+LDA wins on mean accuracy, but is also the *least* consistent across subjects — CSP+LDA, despite a lower mean, is the most stable. A trade-off between peak performance and reliability across a heterogeneous population, typical when comparing methods with different modelling capacity.